In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import base64

In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

In [ ]:
# # PARMS
# changeable
org_id = 1

# fixed
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

In [ ]:

config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

In [ ]:
eeg_selected_feat = raw_eeg[["time", "mp_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.describe())
eeg_selected_feat.head()

In [ ]:
work = eeg_selected_feat[
    (eeg_selected_feat["time"] > pd.Timestamp(datetime(2025, 9, 20), tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(datetime(2025, 9, 29), tz='UTC'))
]
del eeg_selected_feat

# work = eeg_selected_feat[
#     (eeg_selected_feat["time"] >= pd.Timestamp(datetime(2025, 9, 28, 19, 30), tz='UTC')) &
#     (eeg_selected_feat["time"] < pd.Timestamp(datetime(2025, 9, 28, 19, 45), tz='UTC'))
# ]
# del eeg_selected_feat

In [ ]:
work.sum(numeric_only=True)

In [ ]:
sns.histplot(work.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])

In [ ]:
def plot_sorted_mps(df) -> None:
    df = df.sort_values(ascending=False)
    # for the x-axis: sequential numbering
    x = np.arange(1, len(df) + 1)
    y = df.values
    mp_ids = df.index  # for hover

    # Plotly bar chart
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=x,
        y=y,
        hovertext=mp_ids,     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    ))

    fig.update_layout(
        title="Sorted comm_cov values (interactive load-curve style)",
        xaxis_title="Sorted index",
        yaxis_title=df.name,
        template="plotly_white"
    )

    fig.show()

In [ ]:
plot_sorted_mps(work[work["energy_direction"]=="C"].groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])

### Waterfilling Opt

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["mp_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_meas_gen", "wt_surp_gen":"sum_surp_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
agg_on_time.head()

In [ ]:
time_with_deficit = agg_on_time[agg_on_time["sum_surp_gen"] <= 0]
print(f"{len(time_with_deficit)}/{len(agg_on_time)} timestamps has surplus")
time_with_deficit.head()

In [ ]:

work_tf_cons = pd.merge(left=work[work["energy_direction"] == 'C'], right=time_with_deficit, on="time", how="inner")
# INIT
work_tf_cons["A_0"] = 0
work_tf_cons["a_0"] = 0
work_tf_cons["r_0"] = work_tf_cons["wt_meas_cons"]
work_tf_cons["R_0"] = work_tf_cons["sum_meas_gen"]
work_tf_cons["U_0"] = True # will this VZP still receive generation at this t?
work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()["U_0"].rename("U_count_0"), on="time", how="left")

i=0
while (work_tf_cons.groupby(by="time").sum()[f"U_{i}"].rename(f"U_count_{i}").max() > 0) & (work_tf_cons[f"R_{i}"].max() > 0):

    work_tf_cons[f"A_{i}"] = work_tf_cons[f"R_{i}"] / work_tf_cons[f"U_count_{i}"]
    work_tf_cons[f"a_{i+1}"] = work_tf_cons[[f"r_{i}", f"A_{i}"]].min(axis=1)
    work_tf_cons[f"r_{i+1}"] = work_tf_cons[f"r_{i}"] - work_tf_cons[f"a_{i+1}"]

    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"a_{i+1}"].rename(f"sum_a_{i+1}"), on="time", how="left")
    work_tf_cons[f"R_{i+1}"] = work_tf_cons[f"R_{i}"] - work_tf_cons[f"sum_a_{i+1}"]
    work_tf_cons[f"U_{i+1}"] = work_tf_cons[f"r_{i+1}"] > 0
    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"U_{i+1}"].rename(f"U_count_{i+1}"), on="time", how="left")
    i += 1


# sum up for the final distribution
a_cols = [col for col in work_tf_cons.columns if col.startswith("a_")]
work_tf_cons["cc_opt"] = work_tf_cons[a_cols].sum(axis=1).clip(lower=0)
work_tf_cons["pf"] = work_tf_cons["cc_opt"] /  work_tf_cons["wt_meas_cons"] * 100

tf_schedule = work_tf_cons[["time", "mp_id", "pf"]].copy()
tf_schedule["pf"] = tf_schedule["pf"].fillna(100).clip(upper=100)


In [ ]:
check = work_tf_cons[["time", "mp_id", "wt_meas_cons", "comm_cov", "cc_opt", "pf"]]
sns.histplot(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])
plot_sorted_mps(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])

In [ ]:
sns.histplot(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"])
plot_sorted_mps(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"])

In [ ]:
def gini(x):
    total = 0
    for i, xi in enumerate(x[:-1], 1):
        total += np.sum(np.abs(xi - x[i:]))
    return total / (len(x)**2 * np.mean(x))

In [ ]:
print(f"gini on original cc: {gini(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])}")
print(f"gini on optimised cc: {gini(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"])}")

In [ ]:
# bring the quarter-hourly PF schedule to hourly, to obtain a valid final OUTPUT for the first time
tf_schedule['hour'] = tf_schedule['time'].dt.floor('h')

# compute the aggregated pf per mp_id and hour
max_pf = (
    tf_schedule
    .groupby(['mp_id', 'hour'])['pf']
    .transform('max')
)

# overwrite pf
tf_schedule['pf'] = max_pf

# drop the helper column again once no longer needed
tf_schedule.drop(columns='hour', inplace=True)

### Apply pf schedule

In [ ]:
def apply_pf_schedule_to_mps(all_mps: pd.DataFrame, pf_schedule: pd.DataFrame):
    # TODO NOTE!!! this only works on PF for C-MPS
    # TODO NOTE!!! this only works on PF during "Deficit", i dont how the calculations apply/what they do, during surplus   
    
    applied_pfs = pd.merge(left = all_mps, right=pf_schedule, on=["time", "mp_id"], how="outer").merge(right=agg_on_time, on="time", how="left")
    applied_pfs["pf"] = applied_pfs["pf"].fillna(100)
    applied_pfs["pf"] /= 100
    applied_pfs["surp_ratio"] = (applied_pfs["sum_meas_gen"] / applied_pfs["sum_meas_cons"]).replace(np.nan, 1).clip(upper=1) 

    print(f"Original Sums:")
    print(f"\t wt_meas_cons (c): {applied_pfs["wt_meas_cons"].sum():.3f}")
    print(f"\t comm_cov (cc): {applied_pfs["comm_cov"].sum():.3f}")
    old_restnetzbezug = (applied_pfs["wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum())
    print(f"\t -> Restnetzbezug (c - cc): {old_restnetzbezug:.3f}")
    print(f"\t comm_pot: {applied_pfs["comm_pot"].sum():.3f}")
    print(f"\t wt_meas_gen: {applied_pfs["wt_meas_gen"].sum():.3f}")
    print(f"\t wt_surp_gen: {applied_pfs["wt_surp_gen"].sum():.3f}")

    applied_pfs["opt_meas_cons"] = applied_pfs["wt_meas_cons"] * applied_pfs["pf"]
    applied_pfs["opt_comm_cov"] = applied_pfs["comm_cov"] * applied_pfs["pf"]
    applied_pfs["opt_comm_pot"] = applied_pfs["comm_pot"] * applied_pfs["pf"]

    new_calced_agg_on_time = applied_pfs.groupby(by="time").sum().reset_index()[["time", "opt_meas_cons", "opt_comm_cov", "opt_comm_pot"]].rename(columns={"opt_comm_cov":"sum_opt_comm_cov", "opt_comm_pot":"sum_opt_comm_pot", "opt_meas_cons":"sum_opt_meas_cons"})
    applied_pfs = applied_pfs.merge(new_calced_agg_on_time, on="time", how="left")

    applied_pfs["opt_surp_ratio"] = (applied_pfs["sum_meas_gen"] / applied_pfs["sum_opt_meas_cons"]).replace(np.nan, 1).clip(upper=1) 

    applied_pfs["opt_comm_pot"] = applied_pfs["opt_surp_ratio"] * applied_pfs["opt_meas_cons"]
    applied_pfs["opt_comm_cov"] = applied_pfs["opt_surp_ratio"] * applied_pfs["opt_meas_cons"]

    print(f"\nOptimized Sums:")
    print(f"\t opt_meas_cons: {applied_pfs["opt_meas_cons"].sum():.3f} [wt_meas_cons: {applied_pfs["wt_meas_cons"].sum():.3f}, reduction allowed]")
    print(f"\t opt_comm_cov: {applied_pfs["opt_comm_cov"].sum():.3f} [comm_cov: {applied_pfs["comm_cov"].sum():.3f}, reduction NOT! allowed]")
    opt_restnetzbezug = (applied_pfs["opt_meas_cons"].sum() - applied_pfs["comm_cov"].sum())
    print(f"\t -> opt_Restnetzbezug (c* - cc*): {opt_restnetzbezug:.3f}")
    print(f"\t -> real Restnetzbezug: {old_restnetzbezug}")
    print(f"\t -> real Restnetzbezug + opt_comm_cov = wt_meas_cons / {old_restnetzbezug:.3f} + {applied_pfs["opt_comm_cov"].sum():.3f} = {applied_pfs["wt_meas_cons"].sum():.3f}")
    print(f"\t opt_comm_pot: {applied_pfs["opt_comm_pot"].sum():.3f} [comm_pot: {applied_pfs["comm_pot"].sum():.3f}, reduction NOT! allowed]")

    print(f"\t wt_meas_gen: {applied_pfs["wt_meas_gen"].sum():.3f}")
    print(f"\t wt_surp_gen: {applied_pfs["wt_surp_gen"].sum():.3f}")

    sums_on_c_mps = applied_pfs[applied_pfs["energy_direction"] == "C"].groupby(by="mp_id").sum(numeric_only=True)  
    print(f"\nGini on Original cc: {gini(sums_on_c_mps["comm_cov"]):.3f}")
    print(f"Gini on optimized cc: {gini(sums_on_c_mps["opt_comm_cov"]):.3f} [should be lower than original]")

    return applied_pfs.copy()


In [ ]:
applied_pfs = apply_pf_schedule_to_mps(work, tf_schedule)

In [ ]:
check_full_calc = applied_pfs.groupby(by="time").sum()[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc["cc_diff"] = check_full_calc["comm_cov"] - check_full_calc["opt_comm_cov"]
check_full_calc

In [ ]:
check_full_calc = applied_pfs.groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc["cc_diff"] = check_full_calc["comm_cov"] - check_full_calc["opt_comm_cov"]
check_full_calc.sort_values(by="cc_diff", ascending=False)

In [ ]:
single_time_filtered = applied_pfs[
    (applied_pfs["time"] >= pd.Timestamp(datetime(2025, 9, 20, 4, 30), tz='UTC')) &
    (applied_pfs["time"] < pd.Timestamp(datetime(2025, 9, 20, 4, 45), tz='UTC'))
]

check_full_calc = single_time_filtered[single_time_filtered["energy_direction"]=="C"].groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc["cc_diff"] = check_full_calc["comm_cov"] - check_full_calc["opt_comm_cov"]
check_full_calc.sort_values(by="cc_diff", ascending=False)

## Results

In [ ]:
sums_on_mps = applied_pfs[applied_pfs["energy_direction"] == "C"].groupby(by="mp_id").sum(numeric_only=True)
sums_on_mps

In [ ]:
sns.histplot(sums_on_mps["comm_cov"])
plot_sorted_mps(sums_on_mps["comm_cov"])

In [ ]:
sns.histplot(sums_on_mps["opt_comm_cov"])
plot_sorted_mps(sums_on_mps["opt_comm_cov"])

In [ ]:
def plot_stacked_gain_loss(df, col1, col2):
    # sort by col1
    df = df.sort_values(col2, ascending=False)
    
    x = np.arange(1, len(df) + 1)
    base = df[col1]
    compare = df[col2]

    # determine differences
    diff = compare - base

    # separate positive and negative differences
    positive_diff = diff.clip(lower=0)    # green
    negative_diff = (-diff).clip(lower=0) # red

    # the lower part (blue) is ALWAYS min(col1, col2)
    # → i.e. the common range
    common = np.minimum(base, compare)

    # the change portion sits on top
    # → green = positive increase
    # → red   = negative decrease

    fig = go.Figure()

    # --- shared part (always blue) ---
    fig.add_bar(
        x=x,
        y=common,
        name="Base part",
        marker_color="blue",
        hovertemplate="Common: %{y}<extra></extra>"
    )

    # --- negative difference (red) ---
    fig.add_bar(
        x=x,
        y=negative_diff,
        name="col1 > col2 (loss)",
        marker_color="red",
        hovertemplate="Loss: %{y}<extra></extra>"
    )

    # --- positive difference (green) ---
    fig.add_bar(
        x=x,
        y=positive_diff,
        name="col2 > col1 (gain)",
        marker_color="green",
        hovertemplate="Gain: %{y}<extra></extra>"
    )

    fig.update_layout(
        barmode="stack",
        title=f"Stacked Gain/Loss Chart: {col1} vs {col2}",
        xaxis_title="Sorted index",
        yaxis_title="Values",
        template="plotly_white",
        legend_title="Components"
    )

    fig.show()


In [ ]:
def plot_stacked_gain_loss_sortable(df, col1, col2):
    df = df.reset_index()

    # helper function: compute data for one sort order
    def compute_sorted(sort_col):
        df_sorted = df.sort_values(sort_col, ascending=False)

        x = np.arange(1, len(df_sorted) + 1)
        base = df_sorted[col1]
        compare = df_sorted[col2]

        diff = compare - base
        positive_diff = diff.clip(lower=0)
        negative_diff = (-diff).clip(lower=0)
        common = np.minimum(base, compare)

        return {
            "x": x,
            "common": common,
            "neg": negative_diff,
            "pos": positive_diff,
            "mp_id": df_sorted["mp_id"]
        }

    # precompute for each sort order
    data_col1 = compute_sorted(col1)
    data_col2 = compute_sorted(col2)

    # ─────────────────────────────────────────
    # 1) create figure with the first (col1) data
    # ─────────────────────────────────────────
    fig = go.Figure()

    # blue
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["common"],
        marker_color="blue",
        name="Base",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )
    # red
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["neg"],
        marker_color="red",
        name=f"Reductions: {col1} - {col2}",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )
    # green
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["pos"],
        marker_color="green",
        name=f"Gains: {col2} - {col1}",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )

    # ─────────────────────────────────────────
    # 2) buttons that update the EXISTING traces
    # ─────────────────────────────────────────
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                showactive=True,
                y=-0.12,
                x=0.5,
                xanchor="center",
                # yanchor="bottom",
                direction="right",
                font=dict(size=9),
                pad=dict(l=0, r=0, t=0, b=0),
                buttons=[
                    dict(
                        label=f"sorted by {col1}",
                        method="update",
                        args=[
                            {
                                "x": [
                                    data_col1["x"],  # trace 0
                                    data_col1["x"],  # trace 1
                                    data_col1["x"],  # trace 2
                                ],
                                "y": [
                                    data_col1["common"],
                                    data_col1["neg"],
                                    data_col1["pos"],
                                ],
                                "hovertext": [
                                    data_col1["mp_id"],
                                    data_col1["mp_id"],
                                    data_col1["mp_id"]
                                ]
                            },
                            {"title": f"Participation Factor Opt Results (sorted by {col1})"}
                        ]
                    ),
                    dict(
                        label=f"sorted by {col2}",
                        method="update",
                        args=[
                            {
                                "x": [
                                    data_col2["x"],
                                    data_col2["x"],
                                    data_col2["x"],
                                ],
                                "y": [
                                    data_col2["common"],
                                    data_col2["neg"],
                                    data_col2["pos"],
                                ],
                                "hovertext": [
                                    data_col2["mp_id"],
                                    data_col2["mp_id"],
                                    data_col2["mp_id"]
                                ]
                            },
                            {"title": f"Participation Factor Opt Results (sorted by {col2})"}
                        ]
                    )
                ]
            )
        ]
    )

    fig.update_layout(
        barmode="stack",
        title=f"Participation Factor Opt Results (sorted by {col1})",
        xaxis_title="sorted Order",
        yaxis_title=f"kWh",
        template="plotly_white",
        legend=dict(
            orientation="h",
            x=0,
            y=1.1,
            xanchor="left",
            yanchor="top",
        ),
        xaxis_title_standoff=5,
        margin=dict(t=85, b=10)
    )

    fig.show()


In [ ]:
single_time_filtered = applied_pfs[
    (applied_pfs["time"] >= pd.Timestamp(datetime(2025, 9, 23, 2, 0), tz='UTC')) &
    (applied_pfs["time"] < pd.Timestamp(datetime(2025, 9, 23, 2, 15), tz='UTC'))
]

check_full_calc = single_time_filtered[single_time_filtered["energy_direction"]=="C"].groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc["cc_diff"] = check_full_calc["comm_cov"] - check_full_calc["opt_comm_cov"]
check_full_calc.sort_values(by="cc_diff", ascending=False)

plot_stacked_gain_loss_sortable(check_full_calc, "comm_cov", "opt_comm_cov")

In [ ]:
plot_stacked_gain_loss_sortable(sums_on_mps, "comm_cov", "opt_comm_cov")

In [ ]:
def plot_profile_by_category(
    df,
    energy_col_name='sum_wt_meas_gen',
    agg_func_str='median',
    hue_col='weekday',
    extra_col=None,        # e.g. 'temp'
    logo=None, 
):
    if agg_func_str not in ['mean', 'median', 'sum', 'min', 'max', 'std']:
        raise ValueError(f"Unsupported aggregation function: {agg_func_str}")
    
    if hue_col not in df.columns:
        raise ValueError(f"'{hue_col}' is not a column in the DataFrame!")

    df["daytime"] = df.time.dt.strftime("%H:%M")

    # aggregate the energy data by hue_col & daytime
    temp_df = (
        df
        .groupby([hue_col, "daytime"])[energy_col_name]
        .agg(agg_func_str)
        .reset_index()
    )

    unique_cats = sorted(temp_df[hue_col].unique())
    color_list = px.colors.qualitative.Plotly
    colors = {cat: color_list[i % len(color_list)] for i, cat in enumerate(unique_cats)}

    fig = go.Figure()

    # plot the main data (hue_col)
    for cat in unique_cats:
        cat_df = temp_df[temp_df[hue_col] == cat]
        fig.add_trace(go.Scatter(
            x=cat_df['daytime'],
            y=cat_df[energy_col_name],
            mode='lines',
            name=f'{cat} ({agg_func_str})',
            line=dict(color=colors[cat], dash='solid'),
            yaxis='y'
        ))

    # an additional column (if given)
    if extra_col:
        if extra_col not in df.columns:
            raise ValueError(f"'{extra_col}' is not a column in the DataFrame!")

        extra_df = (
            df
            .groupby("daytime")[extra_col]
            .agg(agg_func_str)
            .reset_index()
        )

        # trace (hidden by default, but the axis stays visible!)
        fig.add_trace(go.Scatter(
            x=extra_df['daytime'],
            y=extra_df[extra_col],
            mode='lines',
            name=f'{extra_col} ({agg_func_str})',
            line=dict(color='black', dash='dot'),
            
            yaxis='y2'
        ))

        # make the y2 axis visible (with title)
        fig.update_layout(
            yaxis2=dict(
                title=extra_col,
                overlaying='y',
                side='right',
                showgrid=False,
                visible=True  # <<< HERE: visible, ALWAYS!
            )
        )

    if logo is not None:
        fig.add_layout_image(logo)


    # General layout
    fig.update_layout(
        title=f'Daily profiles by category: {hue_col} ({agg_func_str})',
        xaxis=dict(
            title='Time of day',
            tickangle=45,
            automargin=True,
            tickfont=dict(size=12)
        ),
        yaxis=dict(title=f'{energy_col_name} ({agg_func_str})'),
        legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center'),
        margin=dict(b=80, t=80, l=60, r=80),
        height=600
    )

    fig.show()


In [ ]:
obj_ids_to_filter = [238, 134, 162, 25]

temp = applied_pfs[applied_pfs["energy_direction"] == "C"]
temp = temp[temp["mp_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="mp_id", logo=logo)

---

In [ ]:
check_full_calc

In [ ]:
check.groupby(by="mp_id").sum(numeric_only=True)

In [ ]:
def apply_tf(tf_schedule:dict[int:int], eeg_data:pd.DataFrame) -> pd.DataFrame:

    temp_eeg = eeg_data.copy()
    
    old_comm_cov_ratio = np.sum(temp_eeg["wt_meas_gen"]) / np.sum(temp_eeg["wt_meas_cons"])

    for act_tf_schedule in tf_schedule.items():
        act_org_id = act_tf_schedule[0]
        act_tf = act_tf_schedule[1]/100

        if eeg_data[eeg_data["mp_id"] == act_org_id]["energy_direction"].min() == 'C':
            temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "wt_meas_cons"] *= act_tf
            temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "comm_pot"] *= act_tf
            temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "comm_cov"] *= act_tf
        elif eeg_data[eeg_data["mp_id"] == act_org_id]["energy_direction"].min() == 'G':
            temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "wt_meas_gen"] *= act_tf
            temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "wt_surp_gen"] *= act_tf

    new_comm_cov_ratio = np.sum(temp_eeg["wt_meas_gen"]) / np.sum(temp_eeg["wt_meas_cons"])

    temp_eeg["comm_pot"] = new_comm_cov_ratio * temp_eeg["wt_meas_cons"]
    temp_eeg["comm_cov"] = new_comm_cov_ratio * temp_eeg["wt_meas_cons"]
    
    temp_eeg["comm_cov_ratio_of_records"] = (temp_eeg["comm_pot"] / temp_eeg["wt_meas_cons"]).replace(np.nan, 1)
    temp_eeg.loc[temp_eeg["energy_direction"] == "G", "comm_cov_ratio_of_records"] = np.nan

    return temp_eeg

def get_tf_schedule(eeg_data:pd.DataFrame, constraints, target_comm_cov:int) -> dict[int:int]:
    temp_eeg = eeg_data.copy()
    
    old_comm_cov_ratio = np.sum(temp_eeg["wt_meas_gen"]) / np.sum(temp_eeg["wt_meas_cons"])

    for act_tf_schedule in tf_schedule.items():
        act_org_id = act_tf_schedule[0]
        act_tf = act_tf_schedule[1]/100

        temp_eeg 

        temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "tf_to_apply"] *= target_comm_cov / temp_eeg.loc[temp_eeg["mp_id"] == act_org_id, "comm_cov"]

    new_comm_cov_ratio = np.sum(temp_eeg["wt_meas_gen"]) / np.sum(temp_eeg["wt_meas_cons"])

    temp_eeg["comm_pot"] = new_comm_cov_ratio * temp_eeg["wt_meas_cons"]
    temp_eeg["comm_cov"] = new_comm_cov_ratio * temp_eeg["wt_meas_cons"]
    
    temp_eeg["comm_cov_ratio_of_records"] = (temp_eeg["comm_pot"] / temp_eeg["wt_meas_cons"]).replace(np.nan, 1)
    temp_eeg.loc[temp_eeg["energy_direction"] == "G", "comm_cov_ratio_of_records"] = np.nan

    return temp_eeg

